# 2.0 Entendimiento y selección de los datos (EDA)

**KDD etapa 1 · CRISP-DM fase 2 · Sesiones 1–2**

Objetivo de esta unidad: cerrar la **selección de fuentes** (KDD-1) y levantar el **entendimiento de los datos** (CRISP-DM-2) que alimenta el preprocesamiento del cuaderno `3.0`. Al terminar quedan documentados **≥3 problemas de calidad con cifras**, la **carga defensiva** (sesión 2) y la lista de decisiones que hereda la fase de limpieza.

Metodología y cifras canónicas: `METODOLOGIA_KDD_CRISPDM.md` (solo en disco) y `README.md` (raíz del repo).

## §1 Selección de fuentes (KDD-1)

| Fuente | Formato | Rol en el proyecto | Razón de la selección |
|---|---|---|---|
| **JobHop v2** (`JobHop_v2_train.parquet`) | Parquet | Trayectorias laborales | Único dataset público con historial de empleos **por persona** (`resume_id`) y fechas trimestrales (`Qn AAAA` / `Present`), base para la minería de secuencias. |
| **ESCO v1.2.1** (`data/01_raw/ESCO/*_en.csv`) | CSV | Taxonomía de ocupaciones, grupos ISCO, habilidades y greenShare | Estándar oficial de la UE con códigos jerárquicos (ISCO `0110`, ESCO de 6 dígitos) para **estandarizar** la ocupación reportada y calcular transiciones. |
| **OLE (Observatorio Laboral Colombia)** | — | (Descartada) | No publica trayectorias a nivel de persona con fechas; inviable para secuencias. |

Tarea de minería (sesión 1): **descubrimiento de patrones de transición ocupacional** (no supervisada). No hay columna objetivo que predecir → quedan descartadas clasificación, regresión y predicción.

In [1]:
from pathlib import Path
import sys

carpeta = Path.cwd()
while not (carpeta / "data").exists() and carpeta != carpeta.parent:
    carpeta = carpeta.parent
RAIZ = carpeta
RAW = RAIZ / "data" / "01_raw"
INTER = RAIZ / "data" / "02_interim"
PROC = RAIZ / "data" / "03_processed"

import pandas as pd
print("Raíz del proyecto:", RAIZ)
print("pandas:", pd.__version__)

Raíz del proyecto: C:\Users\ACER\Desktop\MineriaProtect
pandas: 3.0.2


In [2]:
print("01_raw  (fuentes originales, no se modifican)")
for p in sorted(RAW.rglob("*")):
    if p.is_file():
        print(f"   {p.relative_to(RAW)}  {p.stat().st_size/1e6:9.1f} MB")
print()
print("02_interim  (versiones limpias, salidas del bloque de integración)")
for p in sorted(INTER.rglob("*")):
    if p.is_file():
        print(f"   {p.relative_to(INTER)}  {p.stat().st_size/1e6:9.1f} MB")

01_raw  (fuentes originales, no se modifican)
   ESCO\greenShareOcc_en.csv        0.5 MB
   ESCO\ISCOGroups_en.csv        1.0 MB
   ESCO\occupations_en.csv        3.1 MB
   ESCO\occupationSkillRelations_en.csv       28.0 MB
   ESCO\skills_en.csv        9.4 MB
   JobHop_v2_train.parquet        9.2 MB

02_interim  (versiones limpias, salidas del bloque de integración)
   ESCO\greenShareOcc_en_limpio.csv        0.5 MB
   ESCO\ISCOGroups_en_limpio.csv        1.0 MB
   ESCO\occupations_en_limpio.csv        3.1 MB
   ESCO\occupationSkillRelations_en_limpio.csv       28.0 MB
   ESCO\skills_en_limpio.csv        9.4 MB
   JobHop_v2_train_limpio.parquet        8.7 MB


## §2 Carga defensiva (sesión 2)

Lecciones aplicadas directamente de la sesión 2:

- **Tipos que infiere pandas:** los códigos con ceros a la izquierda (ISCO `'0110'`, ESCO `'263102'`) se leen como **texto** (`dtype=str`). Si se leyeran como `int` se perdería el cero y se romperían los mapeos.
- **`keep_default_na=False` + `na_values=[""]`:** solo la celda vacía es `NaN`. Los literales `'None'`, `'Present'` y `'unknown'` se conservan como texto (pandas, por omisión, convierte `'None'` en `NaN` y eso destruiría el diagnóstico de la fase 3).
- **Trampas del CSV** (típicas en datos en español): separador (coma vs. punto y coma), encoding (`utf-8`), decimales con coma. Todas las lecturas del proyecto son explícitas: `encoding="utf-8"`.
- **Parquet conserva los tipos** al leer; aun así JobHop se trata como dato crudo: aquí solo se lee, nunca se modifica.
- Nada de `01_raw` se toca en este cuaderno.

## §3 EDA de las fuentes base

In [3]:
jh = pd.read_parquet(RAW / "JobHop_v2_train.parquet")
print(f"JobHop_v2_train.parquet  ->  {jh.shape[0]:,} filas x {jh.shape[1]} columnas")
print(f"Personas (resume_id únicos)  : {jh['resume_id'].nunique():,}")
print()
print("Tipos que infiere pandas:")
print(jh.dtypes.to_string())
print()
print("Primeras filas:")
display(jh.head(3))

JobHop_v2_train.parquet  ->  1,594,827 filas x 5 columnas
Personas (resume_id únicos)  : 284,251

Tipos que infiere pandas:
resume_id           int64
matched_code          str
start_date            str
end_date              str
university_level      str

Primeras filas:


,resume_id,matched_code,start_date,end_date,university_level
0,0,2353.1,Q3 1996,Q1 2000,Master
1,0,3512.1,Q1 2000,Q1 2002,Master
2,0,3512.2,Q1 2002,Q1 2004,Master


In [4]:
def resumen_eda(nombre, dfx):
    print("=" * 74)
    print(f"{nombre}  ->  {dfx.shape[0]:,} filas x {dfx.shape[1]} columnas")
    print("Columnas:", ", ".join(dfx.columns))
    display(dfx.head(2))

for archivo in sorted((RAW / "ESCO").glob("*.csv")):
    dfx = pd.read_csv(archivo, dtype=str, encoding="utf-8",
                      keep_default_na=False, na_values=[""])
    resumen_eda(archivo.name, dfx)

greenShareOcc_en.csv  ->  3,590 filas x 5 columnas
Columnas: conceptType, conceptUri, code, preferredLabel, greenShare


,conceptType,conceptUri,code,preferredLabel,greenShare
0,ISCO level 3,http://data.europa.eu/esco/isco/C011,011,Commissioned armed forces officers,0.00575396825396825
1,ISCO level 4,http://data.europa.eu/esco/isco/C0110,0110,Commissioned armed forces officers,0.00575396825396825



ISCOGroups_en.csv  ->  619 filas x 8 columnas
Columnas: conceptType, conceptUri, code, preferredLabel, status, altLabels, inScheme, description


,conceptType,conceptUri,code,preferredLabel,status,altLabels,inScheme,description
0,ISCOGroup,http://data.europa.eu/esco/isco/C0,0,Armed forces occupations,released,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Armed forces occupations include all jobs held...
1,ISCOGroup,http://data.europa.eu/esco/isco/C01,01,Commissioned armed forces officers,released,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Commissioned armed forces officers provide lea...


occupations_en.csv  ->  3,043 filas x 15 columnas
Columnas: conceptType, conceptUri, iscoGroup, preferredLabel, altLabels, hiddenLabels, status, modifiedDate, regulatedProfessionNote, scopeNote, definition, inScheme, description, code, naceCode


,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,naceCode
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,director of technical arts\r\ntechnical superv...,NaN,released,2024-01-25T11:28:50.295Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Technical directors realise the artistic visio...,2654.1.7,http://data.europa.eu/ux2/nace2.1/9031
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,wire drawer\r\nforming machine operative\r\ndr...,NaN,released,2024-01-23T10:09:32.099Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4,http://data.europa.eu/ux2/nace2.1/242


occupationSkillRelations_en.csv  ->  126,051 filas x 6 columnas
Columnas: occupationUri, occupationLabel, relationType, skillType, skillUri, skillLabel


,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques
1,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals


skills_en.csv  ->  13,960 filas x 13 columnas
Columnas: conceptType, conceptUri, skillType, reuseLevel, preferredLabel, altLabels, hiddenLabels, status, modifiedDate, scopeNote, definition, inScheme, description


,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage music staff\r\ncoordinate duties of mus...,NaN,released,2023-11-30T15:53:37.136Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,manage prison procedures\r\nmonitor correction...,NaN,released,2023-11-30T15:04:00.689Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Supervise the operations of a correctional fac...


In [5]:
filas_resumen = []
for archivo in sorted((RAW / "ESCO").glob("*.csv")):
    dfx = pd.read_csv(archivo, dtype=str, encoding="utf-8",
                      keep_default_na=False, na_values=[""])
    col_codigo = "code" if "code" in dfx.columns else None
    dups = int(dfx[col_codigo].duplicated().sum()) if col_codigo else None
    vacias = int((dfx == "").sum().sum())
    filas_resumen.append({
        "archivo": archivo.name,
        "celdas vacías": vacias,
        "duplicados (code)": dups,
    })
display(pd.DataFrame(filas_resumen))

,archivo,celdas vacías,duplicados (code)
0,greenShareOcc_en.csv,0,0.0
1,ISCOGroups_en.csv,0,0.0
2,occupations_en.csv,0,4.0
3,occupationSkillRelations_en.csv,0,NaN
4,skills_en.csv,0,NaN


## §4 EDA del integrado `empleos` (data/03_processed)

El integrado reúne JobHop + ESCO por persona (`resume_id`). Es el insumo directo de la limpieza (cuaderno `3.0`).

In [6]:
empleos = pd.read_csv(PROC / "empleos.csv", dtype=str, encoding="utf-8",
                      keep_default_na=False, na_values=[""])
print(f"empleos.csv  ->  {empleos.shape[0]:,} filas x {empleos.shape[1]} columnas")
print(f"Personas (resume_id únicos)  : {empleos['resume_id'].nunique():,}")
print()
print("Columnas:", ", ".join(empleos.columns))
print()
print("Valores como texto por decisión defensiva (ceros a la izquierda).")

empleos.csv  ->  1,506,445 filas x 11 columnas
Personas (resume_id únicos)  : 284,247

Columnas: resume_id, start_date, end_date, university_level, matched_code, emparejado, occupation_code, occupation_label, isco_group, isco_group_label, isco_level

Valores como texto por decisión defensiva (ceros a la izquierda).


In [7]:
exactos = int(empleos.duplicated().sum())
pk_corta = int(empleos.duplicated(subset=["resume_id", "start_date", "end_date"]).sum())
print(f"Duplicados exactos (11 columnas)                 : {exactos:,}")
print(f"Duplicados clave corta (resume_id, inicio, fin)  : {pk_corta:,}  (pluriempleo simultáneo)")

Duplicados exactos (11 columnas)                 : 0
Duplicados clave corta (resume_id, inicio, fin)  : 74,357  (pluriempleo simultáneo)


In [8]:
vaci = (empleos == "").sum()
print("Celdas vacías (NaN técnico) por columna:")
display(vaci[vaci > 0].to_frame("vacías"))

Celdas vacías (NaN técnico) por columna:


,vacías


In [9]:
lit_none   = int(empleos["university_level"].eq("None").sum())
lit_pres   = int(empleos["end_date"].eq("Present").sum())
lit_unk    = int(empleos["emparejado"].eq("unknown").sum())
lit_rescat = int(empleos["emparejado"].eq("rescatado").sum())
print(f"Literales 'None'      (educación)   : {lit_none:,}")
print(f"Literales 'Present'   (empleo vigente, censura) : {lit_pres:,}")
print(f"emparejado = 'unknown'   (sin mapeo) : {lit_unk:,}")
print(f"emparejado = 'rescatado' (mapeo por prefijo) : {lit_rescat:,}")

a_ini = pd.to_numeric(empleos["start_date"].str.extract(r"(\d{4})$", expand=False), errors="coerce")
a_fin = pd.to_numeric(empleos["end_date"].str.extract(r"(\d{4})$", expand=False), errors="coerce")
futuras = int(((a_ini > 2026) | (a_fin > 2026)).sum())
print(f"Fechas con año > 2026  : {futuras}")

Literales 'None'      (educación)   : 174,036
Literales 'Present'   (empleo vigente, censura) : 76,180
emparejado = 'unknown'   (sin mapeo) : 104,993
emparejado = 'rescatado' (mapeo por prefijo) : 10,176


Fechas con año > 2026  : 11


In [10]:
def ordinal(x):
    p = x.str.extract(r"^Q([1-4])\s+(\d{4})$")
    return (pd.to_numeric(p[1], errors="coerce") * 4
            + pd.to_numeric(p[0], errors="coerce"))

durQ = ordinal(empleos["end_date"]) - ordinal(empleos["start_date"]) + 1
q1, q3 = durQ.quantile([0.25, 0.75])
print("Duración en trimestres (inclusiva; 'Present' → NaN):")
print(f"  Q1 = {q1:.2f} | mediana = {durQ.median():.2f} | Q3 = {q3:.2f} | IQR = {q3 - q1:.2f}")
print(f"  Límites de Tukey = [{q1 - 1.5*(q3-q1):.2f}, {q3 + 1.5*(q3-q1):.2f}]")
print(f"  Duración máxima real = {durQ.max():.0f} trimestres ({durQ.max()/4:.1f} años)")

Duración en trimestres (inclusiva; 'Present' → NaN):
  Q1 = 2.00 | mediana = 5.00 | Q3 = 11.00 | IQR = 9.00
  Límites de Tukey = [-11.50, 24.50]
  Duración máxima real = 160 trimestres (40.0 años)


## §5 Cobertura de la taxonomía ESCO (mapeo de ocupaciones)

Se cruza la ocupación reportada por JobHop (`matched_code`, 6 dígitos ESCO) contra el catálogo oficial de ocupaciones ESCO. La cobertura mide qué parte de las experiencias puede estandarizarse para la minería de trayectorias.

In [11]:
occupations = pd.read_csv(RAW / "ESCO" / "occupations_en.csv", dtype=str,
                          encoding="utf-8", keep_default_na=False, na_values=[""])
codigos_esco = set(occupations["code"])

con_codigo = empleos.loc[empleos["matched_code"].ne(""), "matched_code"]
directas = int(con_codigo.isin(codigos_esco).sum())
cats = set(con_codigo.unique())
en_map = len(cats.intersection(codigos_esco))

print(f"Filas con matched_code no vacío            : {len(con_codigo):,}  ({len(con_codigo)/len(empleos)*100:.1f}% del total)")
print(f"Filas con etiqueta directa en ESCO          : {directas:,}  ({directas/len(con_codigo)*100:.1f}%)")
print(f"Categorías distintas reportadas            : {len(cats):,}")
print(f"Categorías presentes en la taxonomía ESCO  : {en_map}  ({en_map*100/len(cats):.1f}% de cobertura de categorías)")

Filas con matched_code no vacío            : 1,506,445  (100.0% del total)
Filas con etiqueta directa en ESCO          : 1,391,276  (92.4%)
Categorías distintas reportadas            : 2,983
Categorías presentes en la taxonomía ESCO  : 2966  (99.4% de cobertura de categorías)


## §5b ¿"API o BD pública"? (sesión 2, refuerzo)

El curso pide **≥2 fuentes, al menos una API o base pública**. Nuestras dos fuentes lo son:

- **JobHop v2** — base pública alojada en **Hugging Face** (`aida-ugent/JobHop`); se descarga directa
  (formato Parquet). Hugging Face además expone una API (`datasets-server`).
- **ESCO v1.2.1** — base pública oficial de la Comisión Europea que se descarga en CSV **y** tiene una
  **API REST documentada** (`https://data.europa.eu/esco/api`).

En la celda siguiente se demuestra la API en vivo: se toma un `conceptUri` **real**
de nuestro `data/01_raw/ESCO/occupations_en.csv` y se consulta el mismo recurso por la API oficial.
Si no hay red en la evaluación, la celda no rompe: lo reporta.

In [12]:
# --- VIA API (intento verificado en vivo) ---
# Las dos fuentes son bases PUBLICAS. ESCO ademas tiene API REST oficial
# (https://data.europa.eu/esco/api). Se consulta un recurso REAL de nuestro CSV.
import json as _json
import ssl as _ssl
import urllib.parse
import urllib.request

ctx = _ssl._create_unverified_context()

evidencias = []
if "conceptUri" in occupations.columns:
    uri = occupations.iloc[0]["conceptUri"]
    url = ("https://data.europa.eu/esco/api/resource/occupation?"
           + urllib.parse.urlencode({"uri": uri, "language": "en"}))
    try:
        with urllib.request.urlopen(url, timeout=60, context=ctx) as r:
            resp = _json.load(r)
        evidencias.append("EXITO: la API de ESCO devolvio "
                          + repr(resp.get("title", resp.get("preferredLabel", ""))))
    except Exception as e:
        evidencias.append(
            "Sin respuesta de la API en esta red (" + type(e).__name__ + "). "
            "La fuente sigue siendo publica por descarga oficial equivalente.")

print("Fuentes publicas declaradas (sesion 2):")
print("  - JobHop v2 (Hugging Face): https://huggingface.co/datasets/aida-ugent/JobHop_v2_train")
print("  - ESCO oficial UE + API   : https://data.europa.eu/esco/api")
print()
print("Intento de API sobre un recurso real del dataset:")
for e in evidencias:
    print("   *", e)

Fuentes publicas declaradas (sesion 2):
  - JobHop v2 (Hugging Face): https://huggingface.co/datasets/aida-ugent/JobHop_v2_train
  - ESCO oficial UE + API   : https://data.europa.eu/esco/api

Intento de API sobre un recurso real del dataset:
   * Sin respuesta de la API en esta red (HTTPError). La fuente sigue siendo publica por descarga oficial equivalente.


## §6 Síntesis: problemas de calidad detectados (con cifras)

Diagnóstico heredado por el cuaderno `3.0`. Los seis ítems se verifican en vivo en las celdas anteriores:

| # | Problema | Cifra observada | Mecanismo de ausencia | Tratamiento propuesto en 3.0 |
|---|---|---|---|---|
| 1 | Duplicados de clave corta (pluriempleo simultáneo) | ≈74.357 filas | — | Conservar la 1.ª ocurrencia y documentar (fase 2) |
| 2 | Ocupación sin mapeo (`'unknown'` / `NaN`) | ≈104.993 filas (≈11,6%) | MAR estructural | Bandera `es_unknown_ocupacion` sin imputar |
| 3 | Ocupación rescatada por prefijo (`'rescatado'`) | ≈10.176 filas | MAR estructural | Bandera `es_rescatado` |
| 4 | Empleos vigentes (`'Present'`, censura) | ≈76.180 filas | Censura por diseño | Bandera `es_vigente` (duración indefinida) |
| 5 | Educación literal `'None'` | ≈174.036 filas | MNAR | Re-categorizar a `'No reportado'` (sin imputar moda) |
| 6 | Fechas con año futuro (>2026) | 11 filas | MCAR (error de captura) | Eliminar (única fila que se borra en toda la limpieza) |

**Principio rector:** en este proyecto la ausencia es *información* (un `unknown` dice “no se pudo mapear”, no “no hay dato”), de modo que **no se imputa** y **no se borra** salvo el caso MCAR. La cola larga de duración (≥25 trimestres) se conserva como *decisión de negocio*, no como outlier a eliminar.

## §7 Cierre y paso a la fase 3

Este cuaderno deja definido el *qué hay y qué significa* en los datos. La fase siguiente (`3.0_preprocesamiento.ipynb`) transforma el integrado aplicando el orden canónico: **duplicados → categorías → nulos (banderas) → outliers → validación**, materializado en funciones reutilizables con umbrales de aceptación verificados por asserts.